<a href="https://colab.research.google.com/github/icosahedron31/Facial-Expression-Recognition/blob/main/CNNMLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install kaggle wandb onnx -Uq


In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
! mkdir ~/.kaggle

mkdir: cannot create directory ‘/root/.kaggle’: File exists


In [29]:
!cp /content/drive/MyDrive/ColabNotebooks/kaggle_API_credentials/kaggle.json ~/.kaggle/kaggle.json

In [30]:
! chmod 600 ~/.kaggle/kaggle.json

In [31]:
!kaggle competitions download challenges-in-representation-learning-facial-expression-recognition-challenge

challenges-in-representation-learning-facial-expression-recognition-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)


In [32]:
! unzip challenges-in-representation-learning-facial-expression-recognition-challenge

Archive:  challenges-in-representation-learning-facial-expression-recognition-challenge.zip
replace example_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: example_submission.csv  
  inflating: fer2013.tar.gz          
  inflating: icml_face_data.csv      
  inflating: test.csv                
  inflating: train.csv               


In [33]:
import wandb
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

**Imports**

In [34]:
import torch # Main PyTorch Library
from torch import nn # Used for creating the layers and loss function
from torch.optim import Adam # Adam Optimizer
import torchvision.transforms as transforms # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import matplotlib.pyplot as plt # Used for visualizing the images and plotting the training progress
from PIL import Image # Used to read the images from the directory
import pandas as pd # Used to read/create dataframes (csv) and process tabular data
import numpy as np # preprocessing and numerical/mathematical operations
import os # Used to read the images path from the directory
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)

Device available:  cuda


**Reading Data**

In [35]:
df = pd.read_csv('../content/train.csv')
df.head()

,emotion,pixels
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...
1,0,151 150 147 155 148 133 111 140 170 174 182 15...
2,2,231 212 156 164 174 138 161 173 182 200 106 38...
3,4,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...
4,6,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...


**Train Test Split**

In [36]:
train_ds, test_ds = train_test_split(df, test_size = 0.3, random_state=42)
test_ds, val_ds = train_test_split(test_ds, test_size = 0.5, random_state=42)
train_ds.head(), val_ds.head()

(       emotion                                             pixels
 14421        0  157 154 157 137 94 99 115 104 120 110 128 98 1...
 9226         4  11 13 17 20 20 21 24 29 28 28 30 33 40 50 57 6...
 994          2  242 250 154 76 80 69 57 53 50 59 55 45 46 49 5...
 28675        0  111 111 112 110 111 111 109 106 99 88 44 68 12...
 4600         3  7 1 4 17 9 11 25 30 33 31 23 38 29 42 56 48 60...,
        emotion                                             pixels
 17147        3  136 136 137 136 138 137 137 135 133 102 52 36 ...
 14040        3  45 26 33 65 47 36 37 35 36 66 88 69 53 38 47 3...
 21306        3  80 78 78 77 76 76 80 115 132 111 93 80 86 92 8...
 22365        3  250 250 250 251 251 251 252 159 58 69 50 44 42...
 22196        1  255 255 255 255 253 255 165 85 62 57 63 72 90 ...)

**Transform**

In [37]:
st = train_ds.iloc[0]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((48, 48)),
    transforms.Normalize([0.5], [0.5])
])
pixels = st["pixels"]
pixels = np.fromstring(pixels, dtype=np.uint8, sep=" ").reshape(48, 48)
pixels = transform(pixels)
pixels

tensor([[[ 0.2314,  0.2078,  0.2314,  ..., -0.5922, -0.7255, -0.3804],
         [ 0.2471,  0.2314,  0.2157,  ..., -0.5686, -0.6941, -0.5686],
         [ 0.2549,  0.2627,  0.1216,  ..., -0.5922, -0.6314, -0.6941],
         ...,
         [ 0.8039,  0.7412,  0.5059,  ...,  0.4902,  0.4039,  0.2078],
         [ 0.7961,  0.6706,  0.3647,  ...,  0.5059,  0.4588,  0.3020],
         [ 0.7490,  0.5686,  0.2471,  ...,  0.5294,  0.4667,  0.4039]]])

**Creating Dataset**

In [38]:
class CustomImageDataset(Dataset):
  def __init__(self, df, transform=None) :
    self.df = df
    self.transform = transform
  def __getitem__(self, index) :
    rows = self.df.iloc[index]
    pixels = rows["pixels"]
    label = torch.tensor(rows["emotion"], dtype=torch.long).to(device)
    pixels = np.fromstring(pixels, dtype=np.uint8, sep=" ").reshape(48, 48)
    pixels = self.transform(pixels).to(device)
    return pixels, label
  def __len__(self) :
    return len(self.df)

In [39]:
train_ds = CustomImageDataset(train_ds, transform)
val_ds = CustomImageDataset(val_ds, transform)

**Models**

In [40]:
import torch
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)

        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()

        self.fc1 = nn.Linear(20 * 20 * 20, 128)
        self.fc2 = nn.Linear(128, 7)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [41]:
import torch
import torch.nn as nn

class MediumNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()

        self.fc1 = nn.Linear(20 * 20 * 20, 20 * 20)
        self.fc2 = nn.Linear(20 * 20, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 7)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)


        return x

In [77]:
import torch
import torch.nn as nn

class BigNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv0 = nn.Conv2d(1, 5, kernel_size = 3, padding=1)
        self.conv1 = nn.Conv2d(5, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=3) #
        self.conv3 = nn.Conv2d(20, 40, kernel_size = 3, padding = 1)
        self.pool = nn.MaxPool2d(2, 2)
        self.avgpool = nn.AvgPool2d(2, 2)
        self.relu = nn.ReLU()
        self.dropout1d = nn.Dropout(0.3)
        self.dropout2d = nn.Dropout2d(0.3)
        self.fc1 = nn.Linear(40 * 10 * 10, 20 * 20)
        self.batchn1 = nn.BatchNorm1d(20 * 20)
        self.fc2 = nn.Linear(20 * 20, 128)
        self.batchn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 64)
        self.batchn3 = nn.BatchNorm1d(64)
        self.fc4 = nn.Linear(64, 7)

    def forward(self, x):
        x = self.relu(self.conv0(x)) #48 * 48 * 5
        x = self.relu(self.conv1(x))
        #x = self.dropout2d(x) #44 * 44 * 10
        x = self.avgpool(x) #22 * 22 * 10
        x = self.relu(self.conv2(x)) #20 * 20 * 20
        x = self.relu(self.conv3(x))
        x = self.dropout2d(x)
        x = self.pool(x) #10 * 10 * 40
        x = x.view(x.size(0), -1)
        x = self.relu(self.batchn1(self.fc1(x)))
        x = self.dropout1d(x)
        x = self.relu(self.batchn2(self.fc2(x)))
        x = self.dropout1d(x)
        x = self.relu(self.batchn3(self.fc3(x)))
        x = self.dropout1d(x)
        x = self.fc4(x)


        return x

**Train Loop**

In [72]:
import copy
def train_loop(train_ds, val_ds, model_name, learning_rate, batch_size, model, EPOCHS=30) :

  train_dataloader = DataLoader(train_ds, batch_size, shuffle=True)
  val_dataloader = DataLoader(val_ds, batch_size, shuffle=True)
  criterion = nn.CrossEntropyLoss()
  best_model = model
  best_acc_val = 0
  optimizer = Adam(model.parameters(), lr = learning_rate)
  for epoch in range(EPOCHS):
    total_acc_train = 0
    total_loss_train = 0
    total_loss_val = 0
    total_acc_val = 0
    model.train()
    for inputs, labels in train_dataloader:
      optimizer.zero_grad()
      outputs = model(inputs)
      train_loss = criterion(outputs, labels)
      total_loss_train += train_loss.item()
      train_loss.backward()

      train_acc = (torch.argmax(outputs, axis = 1) == labels).sum().item()
      total_acc_train += train_acc
      optimizer.step()
    model.eval()
    with torch.no_grad():
      for inputs, labels in val_dataloader:
        outputs = model(inputs)
        val_loss = criterion(outputs, labels)
        total_loss_val += val_loss.item()

        val_acc = (torch.argmax(outputs, axis = 1) == labels).sum().item()
        total_acc_val += val_acc
    if total_acc_val > best_acc_val :
        best_acc_val = total_acc_val
        best_model = copy.deepcopy(model.state_dict())

    print(f'''Epoch {epoch+1}/{EPOCHS}, Train Loss: {round(total_loss_train/train_ds.__len__(), 4)} Train Accuracy {round((total_acc_train)/train_ds.__len__() * 100, 4)}
                Validation Loss: {round(total_loss_val/val_ds.__len__(), 4)} Validation Accuracy: {round((total_acc_val)/val_ds.__len__() * 100, 4)}''')


    wandb.log({
        "epoch": epoch,
        "train_loss": total_loss_train,
        "train_acc": total_acc_train,
        "val_loss": total_loss_val,
        "val_acc": total_acc_val
    })

  return best_model

**Overfitting Small Dataset**

In [44]:
from torch.utils.data import Subset

small_ds = Subset(train_ds, range(100))
model = SimpleNet().to(device)

wandb.init(
    project="fer",
    group="cnnmlp",
    name="overfitting_small_ds"
)
model = train_loop(small_ds, small_ds, "cnnmlp", 0.0001, 8, model)
torch.save(model, "overfitter.pt")
wandb.finish()


RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

**Training Shallow Model**

In [ ]:
from torch.utils.data import Subset

model = SimpleNet().to(device)

wandb.init(
    project="fer",
    group="cnnmlp",
    name="shallow_model_run"
)
model = train_loop(train_ds, val_ds, "cnnmlp", 0.0001, 32, model)
torch.save(model, "shallow_mlp.pt")
wandb.finish()


**Training deeper model**

overfitting small

In [ ]:
from torch.utils.data import Subset

small_ds = Subset(train_ds, range(100))
model = MediumNet().to(device)

wandb.init(
    project="fer",
    group="cnnmlp",
    name="overfitting_small_ds_medium"
)
model = train_loop(small_ds, small_ds, "cnnmlp", 0.0001, 8, model, 50)
torch.save(model, "overfitter.pt")
wandb.finish()


training main

In [ ]:
from torch.utils.data import Subset

model = MediumNet().to(device)

wandb.init(
    project="fer",
    group="cnnmlp",
    name="medium_model_run"
)
model = train_loop(train_ds, val_ds, "cnnmlp", 0.0001, 64, model)
torch.save(model, "medium_mlp.pt")
wandb.finish()


**Bigger with batchnorm**

In [73]:
from torch.utils.data import Subset

small_ds = Subset(train_ds, range(100))
model = BigNet().to(device)

wandb.init(
    project="fer",
    group="cnnmlp",
    name="overfitting_small_ds_big_dropout"
)
model = train_loop(small_ds, small_ds, "cnnmlp", 0.0001, 8, model, 100)
torch.save(model, "overfitter.pt")
wandb.finish()


Epoch 1/100, Train Loss: 0.2613 Train Accuracy 13.0
                Validation Loss: 0.2544 Validation Accuracy: 15.0
Epoch 2/100, Train Loss: 0.2621 Train Accuracy 18.0
                Validation Loss: 0.2541 Validation Accuracy: 16.0
Epoch 3/100, Train Loss: 0.2729 Train Accuracy 16.0
                Validation Loss: 0.255 Validation Accuracy: 16.0
Epoch 4/100, Train Loss: 0.2711 Train Accuracy 12.0
                Validation Loss: 0.2576 Validation Accuracy: 16.0
Epoch 5/100, Train Loss: 0.2615 Train Accuracy 17.0
                Validation Loss: 0.2574 Validation Accuracy: 16.0
Epoch 6/100, Train Loss: 0.2612 Train Accuracy 17.0
                Validation Loss: 0.2541 Validation Accuracy: 16.0
Epoch 7/100, Train Loss: 0.2622 Train Accuracy 16.0
                Validation Loss: 0.2514 Validation Accuracy: 16.0
Epoch 8/100, Train Loss: 0.2568 Train Accuracy 15.0
                Validation Loss: 0.2511 Validation Accuracy: 18.0
Epoch 9/100, Train Loss: 0.2596 Train Accuracy 15.0
     

epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█
train_acc,▁▂▁▁▁▂▁▂▁▁▁▂▂▂▃▂▃▃▂▂▃▃▃▄▅▄▄▅▅▅▆▇▇▆▇█▇▇▇▇
train_loss,████▇▇▇▇▇▇▆▆▇▆▆▆▆▆▆▆▆▅▆▅▅▅▅▅▄▄▄▃▃▃▃▂▁▁▁▁
val_acc,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▅▆▆▇▇▆▇█████████
val_loss,████████████▇▇▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▂▂▂▂▁
epoch,99
train_acc,89
train_loss,9.45126
val_acc,100
val_loss,6.47933


In [78]:
from torch.utils.data import Subset

model = BigNet().to(device)

wandb.init(
    project="fer",
    group="cnnmlp",
    name="big_model_run"
)
model = train_loop(train_ds, val_ds, "cnnmlp", 0.0001, 32, model)
torch.save(model, "medium_mlp.pt")
wandb.finish()


Epoch 1/30, Train Loss: 0.0582 Train Accuracy 23.4226
                Validation Loss: 0.0542 Validation Accuracy: 32.5284
Epoch 2/30, Train Loss: 0.0535 Train Accuracy 31.9019
                Validation Loss: 0.05 Validation Accuracy: 38.2401
Epoch 3/30, Train Loss: 0.0506 Train Accuracy 37.0671
                Validation Loss: 0.0475 Validation Accuracy: 41.9782
Epoch 4/30, Train Loss: 0.0486 Train Accuracy 39.6148
                Validation Loss: 0.0459 Validation Accuracy: 43.766
Epoch 5/30, Train Loss: 0.0471 Train Accuracy 41.9337
                Validation Loss: 0.0445 Validation Accuracy: 45.7163
Epoch 6/30, Train Loss: 0.0453 Train Accuracy 44.5064
                Validation Loss: 0.0435 Validation Accuracy: 47.6202
Epoch 7/30, Train Loss: 0.0438 Train Accuracy 46.7705
                Validation Loss: 0.0427 Validation Accuracy: 48.4328
Epoch 8/30, Train Loss: 0.0423 Train Accuracy 48.9451
                Validation Loss: 0.0419 Validation Accuracy: 49.7562
Epoch 9/30, Train L

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_acc,▁▂▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇█████
train_loss,█▇▇▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
val_acc,▁▃▄▅▆▆▆▇▇▇▇▇█▇██▇█████████████
val_loss,█▆▅▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▅
epoch,29
train_acc,16125
train_loss,361.34267
val_acc,2246
val_loss,203.87488
